In [2]:
# ================================
# 1. IMPORT LIBRARIES
# ================================
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.3f}".format)

# ================================
# 2. LOAD RAW DATA
# ================================
df = pd.read_csv("../data/raw/crop_data.csv")

print("Initial Shape:", df.shape)
df.head()


Initial Shape: (19689, 10)


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.000,56708,2051.400,7024878.380,22882.340,0.796
1,Arhar/Tur,1997,Kharif,Assam,6637.000,4685,2051.400,631643.290,2057.470,0.710
2,Castor seed,1997,Kharif,Assam,796.000,22,2051.400,75755.320,246.760,0.238
3,Coconut,1997,Whole Year,Assam,19656.000,126905000,2051.400,1870661.520,6093.360,5238.052
4,Cotton(lint),1997,Kharif,Assam,1739.000,794,2051.400,165500.630,539.090,0.421


In [3]:
# ================================
# 3. STANDARDIZE COLUMN NAMES
# ================================
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

df.columns


Index(['crop', 'crop_year', 'season', 'state', 'area', 'production',
       'annual_rainfall', 'fertilizer', 'pesticide', 'yield'],
      dtype='object')

In [4]:
# ================================
# 4. BASIC DATA INSPECTION
# ================================
df.info()
df.describe(include="all")


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19689 entries, 0 to 19688
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   crop             19689 non-null  object 
 1   crop_year        19689 non-null  int64  
 2   season           19689 non-null  object 
 3   state            19689 non-null  object 
 4   area             19689 non-null  float64
 5   production       19689 non-null  int64  
 6   annual_rainfall  19689 non-null  float64
 7   fertilizer       19689 non-null  float64
 8   pesticide        19689 non-null  float64
 9   yield            19689 non-null  float64
dtypes: float64(5), int64(2), object(3)
memory usage: 1.5+ MB


,crop,crop_year,season,state,area,production,annual_rainfall,fertilizer,pesticide,yield
count,19689,19689.000,19689,19689,19689.000,19689.000,19689.000,19689.000,19689.000,19689.000
unique,55,NaN,6,30,NaN,NaN,NaN,NaN,NaN,NaN
top,Rice,NaN,Kharif,Karnataka,NaN,NaN,NaN,NaN,NaN,NaN
freq,1197,NaN,8232,1432,NaN,NaN,NaN,NaN,NaN,NaN
mean,NaN,2009.128,NaN,NaN,179926.570,16435941.273,1437.755,24103312.449,48848.353,79.954
std,NaN,6.498,NaN,NaN,732828.676,263056839.813,816.910,94946004.483,213287.355,878.306
min,NaN,1997.000,NaN,NaN,0.500,0.000,301.300,54.170,0.090,0.000
25%,NaN,2004.000,NaN,NaN,1390.000,1393.000,940.700,188014.620,356.700,0.600
50%,NaN,2010.000,NaN,NaN,9317.000,13804.000,1247.600,1234957.440,2421.900,1.030
75%,NaN,2015.000,NaN,NaN,75112.000,122718.000,1643.700,10003847.200,20041.700,2.389


In [5]:
# ================================
# 5. REMOVE INVALID ROWS
# ================================
# Remove rows with zero or negative values
df = df[
    (df["area"] > 0) &
    (df["production"] > 0) &
    (df["yield"] > 0)
]

print("After removing invalid rows:", df.shape)


After removing invalid rows: (19573, 10)


In [6]:
# ================================
# 6. HANDLE MISSING VALUES
# ================================

# Rainfall → fill using state-wise mean
df["annual_rainfall"] = df.groupby("state")["annual_rainfall"] \
                            .transform(lambda x: x.fillna(x.mean()))

# Fertilizer & Pesticide → median (robust to outliers)
df["fertilizer"] = df["fertilizer"].fillna(df["fertilizer"].median())
df["pesticide"] = df["pesticide"].fillna(df["pesticide"].median())

# Drop rows if critical columns still missing
df.dropna(subset=["annual_rainfall"], inplace=True)

print("After handling missing values:", df.shape)


After handling missing values: (19573, 10)


In [7]:
# ================================
# 7. YIELD VALIDATION (VERY IMPORTANT)
# ================================
df["calculated_yield"] = df["production"] / df["area"]

df["yield_difference"] = abs(df["yield"] - df["calculated_yield"])

df[["yield", "calculated_yield", "yield_difference"]].describe()


,yield,calculated_yield,yield_difference
count,19573.000,19573.000,19573.000
mean,80.428,84.338,5.966
std,880.884,926.307,87.899
min,0.004,0.003,0.000
25%,0.605,0.600,0.014
50%,1.038,1.042,0.057
75%,2.410,2.482,0.174
max,21105.000,21105.000,3733.993


In [8]:
# ================================
# 8. FINALIZE YIELD COLUMN
# ================================
# Trust calculated yield (removes human error)
df["yield"] = df["calculated_yield"]

df.drop(columns=["calculated_yield", "yield_difference"], inplace=True)


In [9]:
# ================================
# 9. OUTLIER REMOVAL (IQR METHOD)
# ================================
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return data[(data[column] >= lower) & (data[column] <= upper)]

for col in ["area", "annual_rainfall", "fertilizer", "pesticide", "yield"]:
    df = remove_outliers_iqr(df, col)

print("After outlier removal:", df.shape)


After outlier removal: (9857, 10)


In [10]:
# ================================
# 10. FEATURE SELECTION (NO LEAKAGE)
# ================================
# Keep production ONLY for EDA, NOT for training
features = df.drop(columns=["yield"])
target = df["yield"]


In [11]:
# ================================
# 11. ENCODE CATEGORICAL FEATURES
# ================================
categorical_cols = ["crop", "season", "state"]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

df_encoded.head()


,crop_year,area,production,annual_rainfall,fertilizer,pesticide,yield,crop_Arhar/Tur,crop_Bajra,crop_Banana,crop_Barley,crop_Black pepper,crop_Cardamom,crop_Cashewnut,crop_Castor seed,crop_Coconut,crop_Coriander,crop_Cotton(lint),crop_Cowpea(Lobia),crop_Dry chillies,crop_Garlic,crop_Ginger,crop_Gram,crop_Groundnut,crop_Guar seed,crop_Horse-gram,crop_Jowar,crop_Jute,crop_Khesari,crop_Linseed,crop_Maize,crop_Masoor,crop_Mesta,crop_Moong(Green Gram),crop_Moth,crop_Niger seed,crop_Oilseeds total,crop_Onion,crop_Other Rabi pulses,crop_Other Cereals,crop_Other Kharif pulses,crop_Other Summer Pulses,crop_Peas & beans (Pulses),crop_Potato,crop_Ragi,crop_Rapeseed &Mustard,crop_Rice,crop_Safflower,crop_Sannhamp,crop_Sesamum,crop_Small millets,crop_Soyabean,crop_Sugarcane,crop_Sunflower,crop_Sweet potato,crop_Tapioca,crop_Tobacco,crop_Turmeric,crop_Urad,crop_Wheat,crop_other oilseeds,season_Kharif,season_Rabi,season_Summer,season_Whole Year,season_Winter,state_Arunachal Pradesh,state_Assam,state_Bihar,state_Chhattisgarh,state_Delhi,state_Goa,state_Gujarat,state_Haryana,state_Himachal Pradesh,state_Jammu and Kashmir,state_Jharkhand,state_Karnataka,state_Kerala,state_Madhya Pradesh,state_Maharashtra,state_Manipur,state_Meghalaya,state_Mizoram,state_Nagaland,state_Odisha,state_Puducherry,state_Punjab,state_Sikkim,state_Tamil Nadu,state_Telangana,state_Tripura,state_Uttar Pradesh,state_Uttarakhand,state_West Bengal
1,1997,6637.000,4685,2051.400,631643.290,2057.470,0.706,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
2,1997,796.000,22,2051.400,75755.320,246.760,0.028,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
4,1997,1739.000,794,2051.400,165500.630,539.090,0.457,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
5,1997,13587.000,9073,2051.400,1293074.790,4211.970,0.668,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
6,1997,2979.000,1507,2051.400,283511.430,923.490,0.506,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False

In [12]:
# ================================
# 12. FINAL CLEAN DATASET
# ================================
print("Final dataset shape:", df_encoded.shape)
df_encoded.describe()


Final dataset shape: (9857, 95)


,crop_year,area,production,annual_rainfall,fertilizer,pesticide,yield
count,9857.000,9857.000,9857.000,9857.000,9857.000,9857.000,9857.000
mean,2009.279,6866.647,7624.183,1363.223,927917.485,1735.631,1.061
std,6.321,9440.059,15690.140,544.991,1300990.499,2242.122,0.850
min,1997.000,0.800,1.000,301.300,94.670,0.090,0.003
25%,2004.000,527.000,388.000,987.500,73314.400,134.970,0.502
50%,2010.000,2838.000,2052.000,1313.948,377388.240,735.000,0.809
75%,2015.000,9500.000,7719.000,1643.700,1253601.360,2450.700,1.306
max,2020.000,65412.000,225852.000,2880.200,9356532.480,9534.000,4.778


In [14]:
# ================================
# 13. SAVE PROCESSED DATA
# ================================
df_encoded.to_csv(
    "../data/processed/cleaned_crop_yield_data.csv",
    index=False
)

print("✅ Cleaned dataset saved successfully")


✅ Cleaned dataset saved successfully
